# Naive RAG

Esta libreta implementa un sistema básico de **Generación Aumentada por Recuperación (RAG)**. El objetivo es responder preguntas utilizando información específica de un documento, en lugar de depender únicamente del conocimiento pre-entrenado de un modelo de lenguaje grande (LLM).

### ¿Qué es RAG (Generación Aumentada por Recuperación) en palabras simples?

Generación Aumentada por Recuperación (RAG) es una arquitectura de modelo híbrida que combina las capacidades de recuperación de información con la generación de lenguaje natural de los Large Language Models (LLMs). Su objetivo principal es mejorar la precisión, relevancia y fiabilidad de las respuestas generadas por los LLMs, especialmente en dominios de conocimiento específicos o cuando se requiere información actualizada.

En un sistema RAG, cuando un LLM recibe una consulta, primero utiliza un componente de recuperación para buscar y extraer fragmentos de información relevantes de una base de conocimiento externa (por ejemplo, documentos, bases de datos, web). Estos fragmentos recuperados se utilizan luego como contexto para el componente de generación del LLM, que sintetiza una respuesta coherente y fundamentada en la información proporcionada. Esto mitiga las "alucinaciones" y permite que el modelo acceda a información más allá de sus datos de entrenamiento originales.

In [ ]:
%%bash
pip install -q langchain langchain-community langchain-text-splitters

### 1. Instalación de Librerías Necesarias

Primero, instalamos las librerías de `langchain` que nos permitirán interactuar con modelos de lenguaje, procesar documentos y crear sistemas RAG. También instalamos `langchain-openai` para usar modelos de OpenAI y `langchain-community` para utilidades diversas, y `langchain-text-splitters` para dividir texto.

In [ ]:
!pip install langchain-openai

### 2. Importación de Módulos

Aquí importamos los módulos específicos de Langchain y OpenAI que utilizaremos para cargar documentos, dividir texto y generar embeddings.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

/tmp/ipykernel_10504/1340361629.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [ ]:
from langchain_community.document_loaders import PyPDFLoader


### 3. Montar Google Drive

Para acceder al archivo de texto (`relojes.txt`) que contiene nuestra base de conocimiento, necesitamos montar Google Drive en el entorno de Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 4. Definir la Ruta del Documento

Especificamos la ruta donde se encuentra nuestro archivo de texto en Google Drive. Asegúrate de que esta ruta sea correcta para tu archivo.

In [ ]:
# Reemplaza la ruta con la que copiaste en el paso anterior
path = '/content/drive/MyDrive/llm-notebooks/relojes.txt'

### 5. Carga del Documento

Utilizamos `TextLoader` de `langchain-community` para cargar el contenido de nuestro archivo de texto. Esto convierte el contenido del archivo en un objeto `Document` que Langchain puede procesar.

In [ ]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader(path)
docs = loader.load()

In [ ]:
loader

### 6. División del Documento en Chunks

Los modelos de lenguaje tienen límites en la cantidad de texto que pueden procesar. Dividimos nuestro documento en fragmentos más pequeños (`chunks`) utilizando `RecursiveCharacterTextSplitter`. Esto es crucial para un RAG efectivo, ya que nos permite buscar y pasar solo la información relevante al LLM.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=10
)
chunks = text_splitter.split_documents(docs)

In [ ]:
chunks

[Document(metadata={'source': '/content/drive/MyDrive/llm-notebooks/relojes.txt'}, page_content='El reloj inteligente NovaFit tiene una batería que dura hasta 7 días con un uso normal.'),
 Document(metadata={'source': '/content/drive/MyDrive/llm-notebooks/relojes.txt'}, page_content='Para sincronizar el reloj con el teléfono, debes descargar la aplicación NovaConnect y activar el'),
 Document(metadata={'source': '/content/drive/MyDrive/llm-notebooks/relojes.txt'}, page_content='el Bluetooth.'),
 Document(metadata={'source': '/content/drive/MyDrive/llm-notebooks/relojes.txt'}, page_content='El dispositivo es resistente al agua hasta 50 metros de profundidad, por lo que puedes usarlo al'),
 Document(metadata={'source': '/content/drive/MyDrive/llm-notebooks/relojes.txt'}, page_content='usarlo al nadar.'),
 Document(metadata={'source': '/content/drive/MyDrive/llm-notebooks/relojes.txt'}, page_content='Si la pantalla no enciende, mantén presionado el botón lateral durante 10 segundos para r

### 7. Configuración de API Key para OpenRouter

Configuramos nuestra clave de API de OpenRouter, la cual se guarda de forma segura en los secretos de Colab. Esta clave es necesaria para autenticarnos y usar los modelos de OpenRouter.

In [ ]:
import os
from google.colab import userdata


In [ ]:
# Configura tu llave en los "Secrets" (ícono de llave) de Colab
os.environ["OPENROUTER_KEY"] = userdata.get('OPENROUTER_KEY')

### 8. Inicializar Cliente de OpenAI para OpenRouter

Inicializamos el cliente de OpenAI (que es compatible con la API de OpenRouter) para poder hacer llamadas a sus modelos de lenguaje. Aquí especificamos la URL base de OpenRouter y nuestra clave de API.

In [ ]:
import os
from openai import OpenAI

# Configurar el cliente para OpenRouter
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=userdata.get('OPENROUTER_KEY'), # Correctly retrieve the API key from secrets
)

### 9. Generación de Embeddings (Ejemplo)

Demostramos cómo generar embeddings (representaciones vectoriales numéricas del texto) para algunos fragmentos de texto de ejemplo. Esto es lo que permite a los sistemas RAG medir la similitud semántica entre el texto de la consulta y los documentos.

In [ ]:
chunks_del_pdf = [
    "El procesamiento de lenguaje natural es una rama de la IA.",
    "Los embeddings convierten texto en vectores numéricos."
]

# El ID del modelo en OpenRouter lleva el prefijo 'openai/'
respuesta = client.embeddings.create(
    input=chunks_del_pdf,
    model="thenlper/gte-large"
)


# Extraer los vectores funciona exactamente igual
vectores = [dato.embedding for dato in respuesta.data]
print(f"¡Éxito! Se generaron {len(vectores)} embeddings.")

¡Éxito! Se generaron 2 embeddings.


In [ ]:
vectores

[[-0.034907568246126175,
  0.004245595540851355,
  -0.013898245990276337,
  -0.00046252302126958966,
  0.0030642335768789053,
  -0.0008004669798538089,
  -0.01178155466914177,
  0.021601783111691475,
  -0.01577153429389,
  0.03401396796107292,
  0.043013930320739746,
  0.022631533443927765,
  -0.008132824674248695,
  -0.031902071088552475,
  -0.012444757856428623,
  0.005321724805980921,
  -0.03652572259306908,
  -0.04509245231747627,
  -0.014621861279010773,
  0.023185940459370613,
  -0.015047062188386917,
  0.017982948571443558,
  -0.08037368208169937,
  -0.007848651148378849,
  -0.014917194843292236,
  0.03977637737989426,
  0.05026630312204361,
  -0.0014160933205857873,
  0.06895303726196289,
  0.04752093181014061,
  -0.029324503615498543,
  -0.021973807364702225,
  0.03453141450881958,
  -0.039282944053411484,
  -0.009267264045774937,
  -0.04094289243221283,
  0.017212118953466415,
  -0.04032532498240471,
  0.008699781261384487,
  -0.028842927888035774,
  0.019613169133663177,
  -

### 10. Generación de Embeddings para los Chunks del Documento

Ahora, generamos los embeddings para todos los `chunks` que creamos de nuestro documento original (`relojes.txt`). Cada fragmento de texto tendrá su propio vector numérico.

In [ ]:
# El ID del modelo en OpenRouter lleva el prefijo 'openai/'
respuesta = client.embeddings.create(
    input=[chunk.page_content for chunk in chunks], # Extract page_content from each Document
    model="thenlper/gte-large"
)


# Extraer los vectores funciona exactamente igual
embeddings_del_pdf = [dato.embedding for dato in respuesta.data]
print(f"¡Éxito! Se generaron {len(embeddings_del_pdf)} embeddings.")

¡Éxito! Se generaron 9 embeddings.


In [ ]:
embeddings_del_pdf

[[-0.010926248505711555,
  0.020326366648077965,
  -0.016505412757396698,
  0.013887646608054638,
  -0.010054141283035278,
  0.010239868424832821,
  -0.014749115332961082,
  0.02658102661371231,
  0.005644675809890032,
  0.037514716386795044,
  0.04518567770719528,
  0.019428381696343422,
  -0.0027814486529678106,
  -0.04315203055739403,
  -0.006617043633013964,
  0.021101608872413635,
  -0.018968015909194946,
  -0.05124181881546974,
  -0.007067152764648199,
  0.010693584568798542,
  -0.019570723176002502,
  0.013525222428143024,
  -0.07867443561553955,
  -0.03282010555267334,
  -0.03792722895741463,
  0.017552554607391357,
  0.0067849066108465195,
  -0.027837801724672318,
  0.06129927560687065,
  0.05686956271529198,
  -0.01140675786882639,
  -0.05339762568473816,
  0.04967973008751869,
  -0.06275711953639984,
  -0.028251176699995995,
  -0.010833540000021458,
  0.017854658886790276,
  -0.026240693405270576,
  -0.02081838808953762,
  -0.041781336069107056,
  0.011470912024378777,
  -0.

### 11. Instalación de FAISS

FAISS (Facebook AI Similarity Search) es una librería para una búsqueda eficiente de similitud en grandes bases de datos de vectores. La instalamos para poder crear y buscar en nuestro almacén de vectores.

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 70.4 MB/s eta 0:00:00


### 12. Creación del Almacén de Vectores FAISS

Creamos un `VectorStore` utilizando FAISS, donde almacenamos los embeddings de nuestros `chunks` junto con sus textos originales y metadatos. Esto nos permitirá buscar rápidamente los fragmentos de texto más relevantes para una consulta dada.

In [ ]:
embeddings_model="thenlper/gte-large"

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# Configuramos el modelo de embeddings para futuras búsquedas
embeddings_model = OpenAIEmbeddings(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=userdata.get('OPENROUTER_KEY'),
    model="thenlper/gte-large"
)

# 1. Extraemos los textos de los chunks
texts = [doc.page_content for doc in chunks]
metadatas = [doc.metadata for doc in chunks]

# 2. Creamos el almacén FAISS usando los vectores que ya generaste (embeddings_del_pdf)
# Emparejamos cada texto con su respectivo vector de embedding
text_embedding_pairs = list(zip(texts, embeddings_del_pdf))

vectorstore = FAISS.from_embeddings(
    text_embedding_pairs,
    embeddings_model,
    metadatas=metadatas
)

print("¡FAISS creado con éxito usando los embeddings generados!")

¡FAISS creado con éxito usando los embeddings generados!


Now that the embeddings are stored, you can perform a similarity search. Let's try searching for a relevant document using a query.

### 13. Realizar Búsqueda de Similitud

Definimos una consulta y generamos su embedding. Luego, usamos el almacén de vectores FAISS para encontrar los `k` (en este caso, 1) documentos más similares a nuestra consulta. Esto recupera el contexto relevante.

In [ ]:
# 1. Definimos la consulta
query = "¿Cuánto dura la batería del reloj NovaFit?"

# 2. Generamos el embedding de la consulta manualmente con el cliente que funciona
query_response = client.embeddings.create(
    input=[query],
    model="thenlper/gte-large"
)
query_vector = query_response.data[0].embedding

# 3. Realizamos la búsqueda en FAISS usando el vector directamente
docs_with_score = vectorstore.similarity_search_by_vector(query_vector, k=1)

# 4. Mostrar el resultado
if docs_with_score:
    doc = docs_with_score[0]
    print(f"Documento encontrado: {doc.page_content}")
    print(f"Fuente: {doc.metadata.get('source', 'N/A')}")
else:
    print("No se encontraron documentos similares.")

Documento encontrado: El reloj inteligente NovaFit tiene una batería que dura hasta 7 días con un uso normal.
Fuente: /content/drive/MyDrive/llm-notebooks/relojes.txt


### 14. Generación de Respuesta con LLM (RAG en Acción)

Finalmente, tomamos el documento más relevante encontrado en el paso de búsqueda y lo usamos como contexto para el LLM. El `system_prompt` guía al LLM para que responda la `query` utilizando **únicamente** el `contexto` proporcionado, evitando alucinaciones y asegurando respuestas basadas en nuestra base de conocimiento.

### Paso Final: Generación con LLM
Ahora que podemos recuperar información relevante, vamos a enviarla a un modelo de lenguaje para que responda a nuestra pregunta basándose en el contexto.

In [ ]:
# 1. Preparamos el contexto a partir del documento recuperado
contexto = docs_with_score[0].page_content

# 2. Definimos el sistema y el mensaje del usuario (Prompt Engineering)
system_prompt = "Eres un asistente experto en productos. Responde de forma concisa y utiliza ÚNICAMENTE el contexto proporcionado. Si no sabes la respuesta, di que no la encuentras en el manual."
user_prompt = f"Contexto: {contexto}\n\nPregunta: {query}"

# 3. Llamamos al LLM (usaremos un modelo de chat en OpenRouter que sea gratuito)
completion = client.chat.completions.create(
  model="cohere/north-mini-code:free",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
  ]
)

# 4. Mostramos la respuesta final
print("--- RESPUESTA DEL LLM ---")
print(completion.choices[0].message.content)

--- RESPUESTA DEL LLM ---
Hasta 7 días con un uso normal.
